# Determine file substructure.
## Credentials/
### -Application/
#### --Github/
##### ---Docker/
###### ----fictional-garbonzo/
###### -----private-a
###### -----public-b
###### ----other-repo/
###### -----private-a
###### -----public-b
##### ---Non-Env repo/
###### ----public
###### ----private
###### ----private API keys
###### ----public API keys
#### --Docker/
##### ---fictonal-garbanzo/
###### ----private-a
###### ----public-a
###### ----private-a
###### ----public-b
###### ----private-a
###### ----public-b
#### --CLI/
##### ---some-utility
###### ----username-password combo
##### ---network stuff
###### ----username-password combo
###### ----identifiable info {ip addresses, passwords, etc.
#### --Other/
##### ---other/


In [3]:
import random
import string


def format_credential(
    group_sizes: list[int]     = (5, 4, 5),   # characters per group
    delimiter:   str           = "-",          # separator between groups
    upper_count: int           = 1,            # uppercase letters per group
    digit_count: int           = 0,            # digits per group
    symbol_count:int           = 0,            # symbols per group
    upper_position: str        = "first",      # "first" | "last" | "random" | "spread"
) -> str:
    """
    Builds a format-pattern string from structural parameters.

    Placeholders:
        x  lowercase letter
        X  uppercase letter
        #  digit  (0-9)
        @  ASCII symbol
        -  delimiter (passed through verbatim)

    Args:
        group_sizes:    Number of characters in each group.
                        e.g. (5, 4, 5) → xxxxx-xxxx-xxxxx
        delimiter:      Character(s) placed between groups.
                        e.g. "-", ":", "::" , " "
        upper_count:    How many characters per group are uppercase (X).
        digit_count:    How many characters per group are digits (#).
        symbol_count:   How many characters per group are symbols (@).
        upper_position: Where to place uppercase characters within a group.
                        "first"  → X always leads   e.g. Xxxxx
                        "last"   → X always trails   e.g. xxxxX
                        "random" → X at random slots  e.g. xxXxx
                        "spread" → X evenly distributed e.g. xXxxXx

    Returns:
        A pattern string such as "Xxxxx-Xxxx-Xxxxx"

    Raises:
        ValueError: if reserved placeholder characters are used as the delimiter,
                    or if slot counts exceed the group size.
    """
    RESERVED = set("xX#@")
    if any(c in RESERVED for c in delimiter):
        raise ValueError(f"Delimiter '{delimiter}' must not contain x X # @")

    def _build_group(size: int) -> list[str]:
        total_special = upper_count + digit_count + symbol_count
        if total_special > size:
            raise ValueError(
                f"upper_count({upper_count}) + digit_count({digit_count}) + "
                f"symbol_count({symbol_count}) = {total_special} exceeds "
                f"group size {size}."
            )

        # Start with all lowercase slots
        slots = ["x"] * size

        # ── Place digits and symbols first (positionally agnostic) ──────────
        special_indices = random.sample(range(size), total_special)
        idx = iter(special_indices)

        digit_slots  = [next(idx) for _ in range(digit_count)]
        symbol_slots = [next(idx) for _ in range(symbol_count)]
        upper_slots  = [next(idx) for _ in range(upper_count)]   # remainder

        for i in digit_slots:
            slots[i] = "#"
        for i in symbol_slots:
            slots[i] = "@"

        # ── Place uppercase according to upper_position ──────────────────────
        lower_indices = [i for i in range(size) if slots[i] == "x"]

        match upper_position:
            case "first":
                chosen = sorted(lower_indices)[:upper_count]
            case "last":
                chosen = sorted(lower_indices)[-upper_count:]
            case "random":
                chosen = random.sample(lower_indices, upper_count)
            case "spread":
                # Evenly space across available lowercase positions
                if upper_count > len(lower_indices):
                    chosen = lower_indices
                else:
                    step   = len(lower_indices) / upper_count
                    chosen = [lower_indices[round(i * step)] for i in range(upper_count)]
            case _:
                raise ValueError(
                    f"upper_position must be 'first', 'last', 'random', or 'spread'. "
                    f"Got '{upper_position}'."
                )

        for i in chosen:
            slots[i] = "X"

        return slots

    groups  = [_build_group(size) for size in group_sizes]
    pattern = delimiter.join("".join(g) for g in groups)
    return pattern

In [6]:
# Default — mirrors original behaviour
format_credential()
# → "Xxxxx-Xxxx-Xxxxx"

# Two uppercase per group, spread out
format_credential(upper_count=2, upper_position="spread")
# → "XxxXx-XxxX-XxxXx"

# PIN-style: digits only, no letters
format_credential(group_sizes=(3, 3), upper_count=0, digit_count=3)
# → "###-###"

# Complex credential with symbols
format_credential(
    group_sizes=(4, 4, 4),
    delimiter=":",
    upper_count=1,
    digit_count=2,
    symbol_count=1,
    upper_position="last",
)
# → "##x@:x##X:@x#X"  (positions vary, pattern is deterministic in structure)

# Long passphrase-style, space-delimited
format_credential(group_sizes=(6, 6, 6, 6), delimiter=" ", upper_count=2)
# → "XxXxxx XxXxxx XxXxxx XxXxxx"

'XXxxxx XXxxxx XXxxxx XXxxxx'

In [23]:
import random
import string
import re

# def format_credential():
#     """
#     Returns a format pattern string using placeholders:
#       x = lowercase letter
#       X = uppercase letter
#       # = digit (0-9)
#       @ = special ASCII symbol
#     Delimiters like - are passed through as-is.
#     """
#     return "xxxxx-Xxx##-xxxxx"   # Example: "abcde-Fgh42-ijklm"


def create_credential(length=None, 
                      char_count=None, 
                      include_special_char=True, 
                      formatter=None):
    """
    Generates a credential string based on a format pattern.

    Args:
        length:              Total expected length (optional, used for validation).
        char_count:          Number of non-delimiter characters expected (optional).
        include_special_char: If False, '@' placeholders are replaced with digits.

    Returns:
        A formatted credential string.
    """
    LOWERCASE  = string.ascii_lowercase          # a-z
    UPPERCASE  = string.ascii_uppercase          # A-Z
    DIGITS     = string.digits                   # 0-9
    SYMBOLS    = "!@#$%^&*()_+=-"               # standard ASCII symbols

    pattern = formatter
    result  = []

    for ch in pattern:
        match ch:
            case 'x':
                result.append(random.choice(LOWERCASE))
            case 'X':
                result.append(random.choice(UPPERCASE))
            case '#':
                result.append(random.choice(DIGITS))
            case '@':
                # Honour the include_special_char flag
                result.append(
                    random.choice(SYMBOLS) if include_special_char
                    else random.choice(DIGITS)
                )
            case _:
                # Delimiters (-  :  .  space, etc.) pass through unchanged
                result.append(ch)

    credential = "".join(result)

    # ── Optional validation ──────────────────────────────────────────────────
    if length is not None and len(credential) != length:
        raise ValueError(
            f"Generated credential length {len(credential)} "
            f"does not match requested length {length}."
        )

    non_delimiters = re.sub(r"[-:.\s]", "", credential)
    if char_count is not None and len(non_delimiters) != char_count:
        raise ValueError(
            f"Non-delimiter character count {len(non_delimiters)} "
            f"does not match requested char_count {char_count}."
        )

    return credential


# # ── Quick demo ───────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     for _ in range(5):
#         print(create_credential(include_special_char=True, 
#                                 formatter = format_credential(
#                                                   group_sizes    =  (2, 2, 2),   # characters per group
#                                                   delimiter      = ":",          # separator between groups
#                                                   upper_count    = 1,            # uppercase letters per group
#                                                   digit_count    = 0,            # digits per group
#                                                   symbol_count   = 0,            # symbols per group
#                                                   upper_position = "first",      # "first" | "last" | "random" | "spread")
#                 )
#             )
#         )
        

In [26]:

import random
import string
import re
import os
from datetime import datetime


def save_credential(
    credential: str,
    filename:   str  = "credential",
    directory:  str  = ".",
    append:     bool = False,
    timestamp:  bool = True,
) -> str:
    """
    Writes a credential to a flat file with no extension.

    Args:
        credential: The credential string to save.
        filename:   Base name of the output file (no extension).
                    Defaults to 'credential'.
        directory:  Directory to write into. Created if it does not exist.
                    Defaults to current working directory.
        append:     If True,  add to existing file (one credential per line).
                    If False, overwrite file each time.
        timestamp:  If True, prefix each line with an ISO-8601 timestamp.

    Returns:
        The absolute path of the written file.

    Raises:
        ValueError: if filename contains path separators or an extension.
        OSError:    if the file cannot be written.

    Example output file — 'credential':
        2026-05-19T14:32:01  Bxkrp-Yjmt-Tqwni
        2026-05-19T14:32:08  Mtrqk-Ybp7-Eoshf
    """
    # ── Guard: no extensions, no path separators in filename ─────────────────
    if os.sep in filename or (os.altsep and os.altsep in filename):
        raise ValueError(
            f"filename must be a bare name, not a path. Got '{filename}'."
        )
    if "." in filename:
        raise ValueError(
            f"filename must have no extension. Got '{filename}'."
        )

    os.makedirs(directory, exist_ok=True)

    filepath = os.path.join(directory, filename)
    mode     = "a" if append else "w"

    line = (
        f"{datetime.now().strftime('%Y-%m-%dT%H:%M:%S')}  {credential}\n"
        if timestamp
        else f"{credential}\n"
    )

    with open(filepath, mode, encoding="utf-8") as fh:
        fh.write(line)

    return os.path.abspath(filepath)


# ── Quick demo ───────────────────────────────────────────────────────────────
if __name__ == "__main__":

    cred = create_credential(
        include_special_char = True,
        formatter = format_credential(
            group_sizes    = (2, 2, 2),
            delimiter      = ":",
            upper_count    = 1,
            digit_count    = 0,
            symbol_count   = 0,
            upper_position = "first",
        )
    )

    print(f"Generated : {cred}")

    saved_path = save_credential(
        credential = cred,
        filename   = "credential",
        directory  = ".",
        append     = True,       # accumulate credentials across runs
        timestamp  = True,       # prefix each line with date-time
    )

    print(f"Saved to  : {saved_path}")

Generated : Hp:Zr:Ok
Saved to  : /workspace/Key-Storage/Module-Developer/credential
